# 02. Валидация org-baseline на Colab

Инференс baseline организаторов (`cross-encoder/ms-marco-MiniLM-L12-v2` CLS-эмбеддинги + LogReg) на **полных** `matches.parquet` + `items_human.parquet` (365 654 пары / 711 304 товара).

## Подготовка артефактов (один раз)

1. Локально уже собран архив **`data/baseline/baseline_eval.zip`** (~123 MB): веса модели (safetensors + токенизатор), `baseline_logreg_l12.joblib`, `src/utils.py`.
2. В Colab: **перетащить этот файл в `/content/`** (панель Files → drag&drop). Никаких загрузок на Google Drive не нужно.

Данные подтягиваются напрямую с HF (`hf://`) — нужен только `HF_TOKEN` в секретах Colab (userdata).

In [1]:
%pip install -e .[train]
%pip install -U -q polars scikit-learn

Obtaining file:///content
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build editable did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build editable ... error
error: subprocess-exited-with-error

× Getting requirements to build editable did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [2]:
import json
import os
import sys
import time
import zipfile

import joblib
import numpy as np
import polars as pl
import torch
import dotenv

import utility
from utility.eval import macro_pr_auc

In [3]:
dotenv.load_dotenv()
env = utility.load()
repo_url = f"hf://datasets/{env.config.data.data_repo}"

## 1. Распаковка baseline

In [4]:
ZIP_PATH = "/content/baseline_eval.zip"
BASE_DIR = "/content/baseline"
MODEL_PATH = f"{BASE_DIR}/models/cross-encoder-ms-marco-MiniLM-L12-v2"
LOGREG_PATH = f"{BASE_DIR}/baseline_logreg_l12.joblib"

assert os.path.exists(ZIP_PATH), f"не найден {ZIP_PATH} — перетащите zip в /content"

if not os.path.exists(f"{BASE_DIR}/src/utils.py"):
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall("/content")
print("baseline распакован:", os.listdir(BASE_DIR))

sys.path.insert(0, BASE_DIR)
from src.utils import _extract_embeddings_opt_v2, _load_cross_encoder_opt

baseline распакован: ['baseline_logreg_l12.joblib', 'src', 'models']


## 2. Данные (polars, полный прогон)

Тексты товаров строим в polars ровно как в `utils.py`: `Name: {name} Category: {category} Attributes: {k}: {v} ...`.

In [5]:
items_human = pl.read_parquet(f"{repo_url}/{env.config.data.items_human}")
matches = pl.read_parquet(f"{repo_url}/{env.config.data.matches}")
print("items_human:", items_human.height, "matches:", matches.height)

items_human: 711304 matches: 365654


In [6]:
def _product_text(name, category, attributes):
    try:
        attrs = json.loads(attributes)
        attr_text = " ".join(f"{k}: {v}" for k, v in attrs.items())
    except (TypeError, json.JSONDecodeError):
        attr_text = ""
    return f"Name: {name} Category: {category} Attributes: {attr_text}"

texts = items_human.select(
    "id",
    pl.struct(["name", "category", "attributes"])
    .map_elements(lambda r: _product_text(r["name"], r["category"], r["attributes"][:1500]), return_dtype=pl.String)
    .alias("text"),
)

pairs = (
    matches.join(texts, left_on="id1", right_on="id")
    .rename({"text": "text1"})
    .join(texts, left_on="id2", right_on="id")
    .rename({"text": "text2"})
    .select("id1", "id2", "target", "text1", "text2")
)
print("пар с текстами с обеих сторон:", pairs.height)
pairs.head(2)

пар с текстами с обеих сторон: 365654


id1,id2,target,text1,text2
i64,i64,f64,str,str
781684142473,427,0.0,"""Name: подшипник ступицы передн…","""Name: комплект подшипника ступ…"
833223658747,2639,0.0,"""Name: js asakashi фильтры сало…","""Name: фильтр салонный skoda fa…"


### Проверка заявления организаторов: `items_human ⊆ items`

(по id; читаем только колонку `id` из `items.parquet`, ~1-2 мин по сети)

In [7]:
VERIFY_SUBSET = True

if VERIFY_SUBSET:
    item_ids = pl.scan_parquet(f"{repo_url}/{env.config.data.items}").select("id").collect()
    missing = items_human.select("id").join(item_ids, on="id", how="anti").height
    print(f"товаров items_human без id в items: {missing} (0 = утверждение верно)")
else:
    print("проверка пропущена")

товаров items_human без id в items: 0 (0 = утверждение верно)


## 3. Инференс cross-encoder (CLS-эмбеддинги)

Весёлые детали: `fp16` на T4 (sm_75), `bf16` на Ampere+; batch 512, сортировка по длине (bucketing), warmup. Прогресс — tqdm (как в `utils.py`).

In [8]:
cap = torch.cuda.get_device_capability(0)
dtype = torch.bfloat16 if cap >= (8, 0) else torch.float16
print(f"GPU: {torch.cuda.get_device_name(0)} (sm={cap}), dtype={dtype}")

ce = _load_cross_encoder_opt(MODEL_PATH, dtype=dtype)

pair_list = list(zip(pairs["text1"].to_list(), pairs["text2"].to_list()))
t0 = time.time()
embeddings = _extract_embeddings_opt_v2(ce, pair_list, batch_size=512, warmup=True)
elapsed = time.time() - t0
print(f"эмбеддинги: {embeddings.shape}, {elapsed:.0f} c, {len(pair_list)/elapsed:.0f} пар/с")

GPU: Tesla T4 (sm=(7, 5)), dtype=torch.float16
[CE] device=cuda  dtype=torch.float16  attn=sdpa  backend=pytorch  compile=False(reduce-overhead)  onnx_provider=cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Extracting embeddings: 100%|██████████| 715/715 [23:21<00:00,  1.96s/it]

эмбеддинги: (365654, 384), 1402 c, 261 пар/с


In [9]:
clf = joblib.load(LOGREG_PATH)
scores = clf.predict_proba(embeddings)[:, 1]
print("predict:", scores.shape, "range:", round(float(scores.min()), 4), "-", round(float(scores.max()), 4))

predict: (365654,) range: 0.0 - 1.0


## 4. Оценка: AP + Macro PR-AUC по 20 категориям

Референсы: random = positive rate **0.257**, `name_jaccard` (ours, по EDA) macro **0.319**.

In [10]:
from sklearn.metrics import average_precision_score

y = pairs["target"].to_numpy()
cats = pairs.join(items_human.select("id", "category"), left_on="id1", right_on="id")["category"].to_numpy()

ap_global = average_precision_score(y, scores)
macro = macro_pr_auc(y, scores, cats)
print(f"global AP:    {ap_global:.4f}")
print(f"macro PR-AUC: {macro:.4f}   (random ~0.257, jaccard 0.319)")

global AP:    0.4397
macro PR-AUC: 0.3635   (random ~0.257, jaccard 0.319)


In [11]:
rows = []
for cat in np.unique(cats):
    m = cats == cat
    if y[m].sum() < 2:
        continue
    rows.append((cat, int(m.sum()), int(y[m].sum()), round(float(y[m].mean()), 3), round(float(average_precision_score(y[m], scores[m])), 4)))

per_cat = pl.DataFrame(rows, schema=["category", "n_pairs", "n_pos", "pos_rate", "AP"], orient="row")
per_cat.sort("AP", descending=True)

category,n_pairs,n_pos,pos_rate,AP
str,i64,i64,f64,f64
"""Детские товары""",17561,9869,0.562,0.6643
"""Хобби и творчество""",17677,8202,0.464,0.6188
"""Бытовая химия""",17436,8183,0.469,0.522
"""Красота и гигиена""",17286,6283,0.363,0.4734
"""Бытовая техника""",17814,6404,0.359,0.4665
…,…,…,…,…
"""Автотовары""",19010,3326,0.175,0.2576
"""Мебель""",18366,2756,0.15,0.2097
"""Одежда""",23357,2751,0.118,0.1818


## 5. Выводы

### Итоговые цифры (полный прогон: 365 654 пары, Tesla T4, fp16, batch 512, 23:21 мин, 261 пар/с)

| Метрика | Земля | Значение | Референс |
|---|---|---|---|
| global AP | 0.4397 | random 0.257 |
| **macro PR-AUC** | **0.3635** | random 0.257, name_jaccard 0.319 |

### Категории (macro-метрика, AP по каждой)

- **Силён** там, где много дублей: Детские товары 0.664 (pos_rate 0.562), Хобби и творчество 0.619 (0.464), Бытовая химия 0.522 (0.469).
- **Провален** там, где дублей мало и специфичная лексика: Ювелирные изделия 0.112 (pos_rate 0.124), Обувь 0.135 (0.099), Одежда 0.182 (0.118), Мебель 0.210 (0.150), Автотовары 0.258 (0.175).

### Интерпретация

1. Cross-encoder даже на английской модели (MS-Marco) существенно обгоняет лексику: macro +0.04 над jaccard, global AP +0.18 над baseline.
2. Разброс по категориям огромен — слабые категории (ювелирка, одежда, обувь) с малым pos_rate и специфической русской лексикой — кандидаты на основной прирост от RU-модели и дообучения.
3. Скорости достаточно: 261 пар/с на T4 → на H100 ≈ 0.8-1K пар/с; лимиты Check 1 мин (1000 пар) / Public 6 мин (115K) / Private 13 мин (275K) — запас есть.

### Гипотезы для нашей модели

- RU-backbone (ruBert / ruMiniLM / sbert_nlu_ru) — закрыть языковой gap (особенно одежда/обувь/ювелирка).
- Дообучение cross-encoder на 11.2M llm-таргетах (soft BCE), валидация на human-парах тем же `macro_pr_auc`.
- Полный текст пары с атрибутами уже учтён (обрезка атрибутов до 1500 символов — безопасна).